In [ ]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

from diffusers import StableDiffusionPipeline
from torchvision import transforms

from torchmetrics.image.fid import FrechetInceptionDistance
from prdc import compute_prdc
import random


In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

pipe.load_lora_weights(
    "lora_output/new_experiment",
    weight_name="last.safetensors"
)

pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()


In [ ]:
import open_clip

device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
)

model = model.to(device)
model.eval()

In [ ]:
import numpy as np

def extract_biomedclip_features(images):

    feats = []

    with torch.no_grad():
        for img in images:

            x = preprocess(img).unsqueeze(0).to(device)

            f = model.encode_image(x)

            f = f / f.norm(dim=-1, keepdim=True)

            feats.append(f.cpu().numpy())

    return np.vstack(feats)

In [ ]:
from scipy import linalg

def compute_fid_from_features(real_feats, fake_feats):

    mu1 = real_feats.mean(axis=0)
    mu2 = fake_feats.mean(axis=0)

    sigma1 = np.cov(real_feats, rowvar=False)
    sigma2 = np.cov(fake_feats, rowvar=False)

    diff = mu1 - mu2

    covmean = linalg.sqrtm(sigma1 @ sigma2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)

    return float(fid)

In [ ]:
def compute_clip_similarity(fake_imgs, text):

    text_tokens = clip.tokenize([text]).to(device)

    with torch.no_grad():
        text_feat = clip_model.encode_text(text_tokens)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

    sims = []

    for img in fake_imgs:

        x = clip_preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():
            img_feat = clip_model.encode_image(x)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)

        sim = (img_feat @ text_feat.T).item()

        sims.append(sim)

    return np.mean(sims)

In [ ]:
def compute_prd(real_imgs, fake_imgs):
    real_feats = extract_biomedclip_features(real_imgs)
    fake_feats = extract_biomedclip_features(fake_imgs)

    return compute_prdc(
        real_features=real_feats,
        fake_features=fake_feats,
        nearest_k=5
    )


In [ ]:
METADATA = Path("/home/project/datasets/BCN_new/metadata.csv")
IMG_DIR = Path("/home/project/datasets/BCN_new")

df = pd.read_csv(METADATA)

In [ ]:
def generate_images(pipe, cls, n):

    prompt = f"<lesion> dermoscopy image of {PROMPT_MAP[cls]}"
    negative = "clinical photo, ruler, text, watermark, low quality"

    images = []

    for _ in range(n):
        with torch.no_grad(), torch.cuda.amp.autocast():
            img = pipe(
                prompt,
                negative_prompt=negative,
                num_images_per_prompt=1,
                guidance_scale=7.0,
                num_inference_steps=40
            ).images[0]

        images.append(img)

    return images

In [ ]:
def load_real_images(df, image_dir, cls, n=5):

    df_cls = df[df["class"] == cls]

    if len(df_cls) == 0:
        return []

    df_cls = df_cls.sample(n=min(n, len(df_cls)), random_state=42)

    images = []

    for isic_id in df_cls["isic_id"]:
        img_path = Path(image_dir) / f"{isic_id}.jpg"

        if img_path.exists():
            images.append(Image.open(img_path).convert("RGB"))

    print(f"Loaded images → real: {len(images)}")

    return images

In [ ]:
MIN_REAL = 100
N_GEN = 100

results = []

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Dermatofibroma": "DF",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

PROMPT_MAP = {
    "AK": "Actinic keratosis",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis lesion",
    "DF": "Dermatofibroma",
    "MEL": "Melanoma",
    "NV": "Nevus",
    "SCC": "Squamous cell carcinoma",
}

df = pd.read_csv(METADATA)
df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)

# удаляем все строки без класса
df = df.dropna(subset=["class"])
classes = sorted(df["class"].unique())

print(f"Found {len(classes)} classes: {classes}")

for cls in classes:

    print(f"\n=== {cls} ===")

    real_imgs = load_real_images(df, IMG_DIR, cls, N_GEN)

    if len(real_imgs) < 2:
        continue

    fake_imgs = generate_images(pipe, cls, N_GEN)

    real_feats = extract_biomedclip_features(real_imgs)
    fake_feats = extract_biomedclip_features(fake_imgs)

    fid = compute_fid_from_features(real_feats, fake_feats)
    prd = compute_prd(real_imgs, fake_imgs)

    clip_sim = compute_clip_similarity(
        fake_imgs,
        f"dermoscopy image of {PROMPT_MAP[cls]}"
    )

   
    results.append({
        "class": cls,
        "n_real": len(real_imgs),
        "fid": fid,
        "precision": prd["precision"],
        "recall": prd["recall"],
        "clip": clip_sim,
    })

    print(f"FID: {fid:.2f}")
    print(f"Precision: {prd['precision']:.3f}")
    print(f"Recall: {prd['recall']:.3f}")
    print(f"CLIP: {clip_sim:.3f}")

    del fake_imgs
    torch.cuda.empty_cache()
